# SpiritSafe Module Quick Start

The `spirit_safe` module provides high-level APIs for working with the SpiritSafe profile registry: listing profiles, loading configurations, hydrating lookup tables with SPARQL results, traversing profile relationships, and creating multi-entity curation packets.

This notebook demonstrates all major routes with both offline examples and optional live queries.

## Imports and Setup

Start by importing the main components from the spirit_safe module.

In [ ]:
from gkc.spirit_safe import (
    LookupCache,
    LookupFetcher,
    create_curation_packet,
    get_profile_graph,
    get_profile_metadata,
    get_spirit_safe_source,
    hydrate_profile_lookups,
    list_profiles,
    load_manifest,
    load_profile,
    load_profile_package,
    profile_exists,
    resolve_profile_link,
    resolve_profile_path,
    resolve_query_ref,
    set_spirit_safe_source,
    validate_packet_structure,
)

print("SpiritSafe module imports successful")

## Source Configuration

Configure where the spirit_safe module should load profiles from. By default, it uses the GitHub repository.

In [ ]:
# Set GitHub as the source (default)
set_spirit_safe_source(mode="github", github_repo="skybristol/SpiritSafe", github_ref="main")

# Get the current source configuration
source = get_spirit_safe_source()
print(f"Mode: {source.mode}")
print(f"Repository: {source.github_repo}")
print(f"Reference: {source.github_ref}")

## Registry Discovery

List available profiles and check their metadata.

In [ ]:
# List all available profiles
profiles = list_profiles()
print(f"Available profiles: {profiles}")

In [ ]:
# Check if a specific profile exists
if profile_exists("TribalGovernmentUS"):
    metadata = get_profile_metadata("TribalGovernmentUS")
    print(f"Profile: {metadata.name}")
    print(f"Version: {metadata.version}")
    print(f"Status: {metadata.status}")
else:
    print("Profile not found")

## Path Resolution

Resolve the file system paths where profiles and their associated queries are located.

In [ ]:
# Resolve the path to a profile
profile_path = resolve_profile_path("TribalGovernmentUS")
print(f"Profile path: {profile_path}")

# Resolve a query file reference relative to a profile
query_path = resolve_query_ref(
    "queries/wikidata_language_items_en.sparql",
    profile_path
)
print(f"Query path: {query_path}")

## Lookup Hydration

Populate lookup tables with results from SPARQL queries. This enables profile-driven choice lists for data entry.

In [ ]:
# Initialize a lookup cache and fetcher
cache = LookupCache()
fetcher = LookupFetcher(cache=cache)

print(f"Cache initialized: {cache}")
print(f"Fetcher initialized: {fetcher}")

In [ ]:
# Hydrate lookups for profiles (dry run to avoid network calls)
# This scans profiles for lookup specifications and shows what would be fetched
summary = hydrate_profile_lookups(
    [resolve_profile_path("TribalGovernmentUS")],
    dry_run=True,
)

print(f"Profiles scanned: {summary['profiles_scanned']}")
print(f"Lookup specs found: {summary['lookup_specs_found']}")
print(f"Unique queries: {summary['unique_queries']}")

## Manifest Operations

The manifest is the authoritative registry of all profiles and their relationships. Load and query it for registry-level information.

In [ ]:
# Load the registry manifest
manifest = load_manifest()

print(f"Manifest generated at: {manifest.generated_at}")
print(f"Commit SHA: {manifest.commit_sha}")
print(f"Profile IDs in registry: {manifest.profile_ids}")

In [ ]:
# Get details about a specific profile from the manifest
profile_entry = manifest.get_profile_entry("TribalGovernmentUS")
print(f"Profile entry: {profile_entry}")

## Profile Loading

Load individual profiles or multi-profile packages for data curation.

In [ ]:
# Load a single profile YAML
profile = load_profile("TribalGovernmentUS")

print(f"Profile name: {profile.get('name')}")
print(f"Profile keys: {list(profile.keys())}")

In [ ]:
# Load a profile package (primary profile + related profiles)
# depth=1 includes directly linked profiles
package = load_profile_package("TribalGovernmentUS", depth=1)

print(f"Primary profile: {package['primary_profile']}")
print(f"Related profiles in package: {list(package['profiles'].keys())}")
print(f"Package metadata keys: {list(package['metadata'].keys()) if 'metadata' in package else 'No metadata'}")

## Profile Graph Traversal

Navigate relationships between profiles using the profile graph.

In [ ]:
# Load the profile relationship graph
graph = get_profile_graph()

print(f"Total profiles in graph: {graph.profile_count()}")
print(f"Graph edges: {
    len(
        list(
            graph.get_edges(
                source_profile="TribalGovernmentUS"
            )
        )
    ) if hasattr(graph, 'get_edges') else 'N/A'}")

In [ ]:
# Get edges (relationships) involving a specific profile
source_profile = "TribalGovernmentUS"
edges = graph.get_edges(source_profile=source_profile)
for edge in edges:
    print(f"  {source_profile} --[{edge.relationship_type}]--> {edge.target_profile} (via {edge.via_statement})")

In [ ]:
# Resolve a specific profile linkage
# This returns the relationship definition
linkage = resolve_profile_link("TribalGovernmentUS", "office_held_by_head_of_state")
print(f"Linkage: {linkage}")

## Curation Packets

Curation packets are data structures made up of existing content called up from known sources such as Wikidata, passed through and validated with the GKC Entity Profiles, and assembled for data curation activities. They are designed to provide everything that is editable in and around a given entity. The GKC Wizard was the first utility the Curation Packets were designed for, but they can be used for any number of data curation workflows and presented back to the GKC for processing.

The following sections demonstrate the Curation Packet functionality, starting with creating a packet that bundles a primary profile with related profiles.

In [ ]:
# Create a curation packet for a specific profile
# This packages the profile with related profiles for multi-entity curation
packet = create_curation_packet(
    profile_id="TribalGovernmentUS",
    operation_mode="bulk",
    depth=1,
)

print(f"Packet ID: {packet['packet_id']}")
print(f"Operation mode: {packet['operation_mode']}")
print(f"Profiles included: {', '.join([e['profile'] for e in packet['entities']])}")

### Packet Validation

Validate that a curation packet is well-formed and ready for execution.

In [ ]:
# Validate the packet structure
is_valid, errors = validate_packet_structure(packet)

print(f"Packet valid: {is_valid}")
if errors:
    print(f"Validation errors: {errors}")
else:
    print("No validation errors")

### Complete Workflow Example

Combine multiple operations for a realistic multi-entity curation scenario.

In [ ]:
def complete_curation_workflow(profile_id: str, depth: int = 1):
    """
    Demonstrate a complete workflow:
    1. Check profile exists
    2. Get metadata
    3. Load manifest
    4. Get profile graph edges
    5. Create curation packet
    6. Validate packet
    """
    
    print(f"\n--- Curation Workflow for {profile_id} ---\n")
    
    # Step 1: Check profile
    if not profile_exists(profile_id):
        print(f"Profile {profile_id} not found")
        return None
    
    # Step 2: Get metadata
    metadata = get_profile_metadata(profile_id)
    print(f"Profile: {metadata.name} (v{metadata.version})")
    
    # Step 3: Get manifest
    manifest = load_manifest()
    print(f"Registry has {len(manifest.profile_ids)} profiles")
    
    # Step 4: Check relationships
    graph = get_profile_graph()
    edges = graph.get_edges(source_profile=profile_id)
    print(f"Profile has {len(edges)} outgoing relationships")
    
    # Step 5: Create packet
    packet = create_curation_packet(
        profile_id=profile_id,
        operation_mode="bulk",
        depth=depth,
    )
    print(f"Created packet {packet['packet_id']}")
    
    # Step 6: Validate
    is_valid, errors = validate_packet_structure(packet)
    print(f"Packet valid: {is_valid}")
    
    return packet

# Run the workflow
result_packet = complete_curation_workflow("TribalGovernmentUS", depth=1)

## Next Steps

With a validated curation packet, you can:

- **Submit for review**: Hand off the packet to a data curator or administrator
- **Generate forms**: Use the Wizard Engineer components to create data entry forms from the profiles
- **Validate data**: Use the Validation Agent to check incoming data against profile constraints
- **Create curation interface**: Build a UI that guides users through multi-entity entry workflows

See the [API documentation](../gkc/api/spirit_safe.md) for complete route reference and the [architecture guide](../architecture/spirit_safe_models.md) for design details.